# Laboratorio de Apache Hive (MetaHive)

Asegúrate de haber inicializado primero los clústeres corriendo tanto `hadoop_infra.ipynb` como `hive_infra.ipynb` antes de ejecutar este laboratorio.


In [1]:
HIVE_USERDIR = "/user/hive"
HIVE_DATADIR = f"{HIVE_USERDIR}/warehouse"

## 1. Preparar un dataset en HDFS

Vamos a simular un archivo de ventas y lo subiremos a HDFS. Hive necesita que los datos estén alojados en HDFS para poder leerlos transparentemente.


In [2]:
import csv
import random
import os

dataset_filename = "sales.csv"
with open(dataset_filename, "w", newline="") as f:
    writer = csv.writer(f)
    # Hive puede ignorar headers fácilmente, pero para simplificar la primera tabla, lo generaremos sin cabecera.
    # Formato esperado: id, producto, categoria, monto, pais
    for i in range(1, 50001):
        writer.writerow(
            [
                i,
                f"Producto_{random.randint(1, 20)}",
                random.choice(["Electronica", "Hogar", "Deportes"]),
                round(random.uniform(10.0, 500.0), 2),
                random.choice(["MX", "US", "CO", "ES"]),
            ]
        )
print(f"¡Dataset '{dataset_filename}' generado localmente con 500,000 filas!")

¡Dataset 'sales.csv' generado localmente con 500,000 filas!


In [3]:
# Pasamos el archivo del host al contenedor del namenode temporalmente, y luego lo consagramos en HDFS
!docker exec namenode hdfs dfs -mkdir -p {HIVE_DATADIR}/external/lab_db/sales
!docker cp sales.csv namenode:/tmp/sales.csv
!docker exec namenode hdfs dfs -put -f /tmp/sales.csv {HIVE_DATADIR}/external/lab_db/sales

## 2. Conexión a Hive via Beeline

Apache Hive procesa consultas SQL (conocido como HQL) y el Metastore convierte esas declaraciones en grafos de ejecución de MapReduce o Tez. Nos conectaremos al motor vía `beeline`.


In [4]:
from pyhive import hive
import pandas as pd

# 1. Establecer la conexión (aquí no especificamos database porque queremos verlas todas)
conexion = hive.Connection(host="localhost", port=10000, username="hive")

# 2. Ejecutar la consulta para listar bases de datos
df_databases = pd.read_sql("SHOW DATABASES", conexion)

# 3. Mostrar el resultado
display(df_databases)

# 4. Cerrar la conexión
conexion.close()

C:\Users\Marco\AppData\Local\Temp\ipykernel_41192\2528582864.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_databases = pd.read_sql("SHOW DATABASES", conexion)


,database_name
0,default


## 3. Creación de Base de Datos y una EXTERNAL TABLE

Las tablas **Externas** en Hive mapean un esquema SQL encima de un directorio puro de HDFS. Si ejecutas un `DROP TABLE` sobre una tabla externa, Hive tira su esquema a la basura pero ¡respeta tus valiosos archivos en el sistema HDFS!


In [5]:
from pyhive import hive
import pandas as pd

# 1. Establecemos la conexión al servidor (sin especificar base de datos inicialmente)
# El usuario "hive" es el estándar, lo agregamos para emular el '-n hive' de beeline
conexion = hive.Connection(host="localhost", port=10000, username="hive")
cursor = conexion.cursor()

# 2. Borrar y crear la base de datos
print("Recreando la base de datos 'lab_db'...")
cursor.execute("DROP DATABASE IF EXISTS lab_db CASCADE")
cursor.execute("CREATE DATABASE IF NOT EXISTS lab_db")

# 3. Le indicamos a la sesión que use 'lab_db' (equivale a conectarse a /lab_db)
cursor.execute("USE lab_db")

# 4. Crear la tabla externa (Aprovechamos las triples comillas de Python para que sea más legible)
# Asumo que HIVE_DATADIR ya lo tienes definido como variable en tu libreta
create_table_sql = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS sales (
    id INT, 
    producto STRING, 
    categoria STRING, 
    monto DOUBLE, 
    pais STRING
)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION 'hdfs://namenode.mavasbel.vpn.itam.mx:8020{HIVE_DATADIR}/external/lab_db/sales'
"""

print("Creando la tabla externa 'sales'...")
cursor.execute(create_table_sql)

# 5. Mostrar las tablas creadas
print("Tablas disponibles en lab_db:")
# Usamos pandas aquí para que la tabla de resultados se vea bonita en tu Jupyter Notebook
df_tablas = pd.read_sql("SHOW TABLES", conexion)
display(df_tablas)

# 6. Cerramos la conexión
conexion.close()

Recreando la base de datos 'lab_db'...
Creando la tabla externa 'sales'...
Tablas disponibles en lab_db:


C:\Users\Marco\AppData\Local\Temp\ipykernel_41192\3821488762.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tablas = pd.read_sql("SHOW TABLES", conexion)


,tab_name
0,sales


## 4. Consulta Directa (Fetch Task)

Un `SELECT *` no requiere cálculo distribuido agresivo. Hive lo identifica y solo pide a HDFS el volcado del archivo como un simple visor de tabla.


In [6]:
from pyhive import hive
import pandas as pd

# 1. Establecemos la conexión apuntando directamente a 'lab_db'
conexion = hive.Connection(
    host="localhost", port=10000, username="hive", database="lab_db"
)

# 2. Definimos la consulta
query = "SELECT * FROM sales LIMIT 5"

# 3. Ejecutamos la consulta y la guardamos en un DataFrame
df_muestra = pd.read_sql(query, conexion)

# 4. Mostramos los 5 registros
display(df_muestra)

# 5. Cerramos la conexión
conexion.close()

C:\Users\Marco\AppData\Local\Temp\ipykernel_41192\4265226897.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_muestra = pd.read_sql(query, conexion)


,sales.id,sales.producto,sales.categoria,sales.monto,sales.pais
0,1,Producto_17,Electronica,106.72,MX
1,2,Producto_3,Hogar,417.95,ES
2,3,Producto_11,Electronica,57.33,ES
3,4,Producto_6,Deportes,394.81,CO
4,5,Producto_18,Hogar,151.85,US


## 5. Consultas con Agregación (Activando procesamiento)

Al usar funciones complejas o `GROUP BY`, notarás cómo Beeline comienza a imprimir los logs en pantalla, porque detrás del SQL, Hive acaba de despachar un Job al ResourceManager.


In [7]:
from pyhive import hive
import pandas as pd

# 1. Establecer la conexión a la base de datos 'lab_db'
conexion = hive.Connection(
    host="localhost", port=10000, username="hive", database="lab_db"
)

# 2. Definir tu consulta de agregación (usamos triples comillas para que se vea ordenada)
query_agg = """
SELECT 
    categoria, 
    pais, 
    COUNT(*) AS total_ventas, 
    ROUND(AVG(monto), 2) AS ticket_promedio, 
    MAX(monto) as venta_maxima 
FROM sales 
GROUP BY categoria, pais 
ORDER BY total_ventas DESC
"""

# 3. Ejecutar la consulta y guardar el resultado en un DataFrame
df_resumen = pd.read_sql(query_agg, conexion)

# 4. Mostrar el DataFrame resultante
display(df_resumen)

# 5. Cerrar la conexión
conexion.close()

C:\Users\Marco\AppData\Local\Temp\ipykernel_41192\397477868.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_resumen = pd.read_sql(query_agg, conexion)


,categoria,pais,total_ventas,ticket_promedio,venta_maxima
0,Electronica,MX,4330,251.67,499.99
1,Deportes,CO,4221,256.99,499.84
2,Electronica,ES,4194,254.37,499.64
3,Electronica,US,4194,253.04,499.86
4,Hogar,ES,4175,254.52,499.80
5,Hogar,US,4166,255.92,499.91
6,Electronica,CO,4162,258.07,499.66
7,Deportes,ES,4160,256.20,499.99
8,Deportes,MX,4129,253.21,499.92
9,Hogar,MX,4107,252.43,499.88


## 6. Manejo Predictivo: Particionamiento (Managed Tables)

El poder de MetaHive en Big Data surge de saltar la lectura física de petabytes enteros de datos. Transformaremos nuestra tabla plana a una **Partitioned Table**. Físicamente agrupará la data por país, lo que nos permitiría consultar "Ventas en MX" en microsegundos porque ¡ignoraría físicamente las carpetas de los demás países!


In [8]:
from pyhive import hive

# 1. Conectar a la base de datos lab_db
conexion = hive.Connection(
    host="localhost", port=10000, username="hive", database="lab_db"
)
cursor = conexion.cursor()

# 2. Habilitar particionado dinámico (esto es CRUCIAL para el INSERT que vas a hacer)
# Sin esto, Hive te pedirá especificar cada partición manualmente.
# print("Configurando entorno para particiones dinámicas...")
# cursor.execute("SET hive.exec.dynamic.partition = true")
# cursor.execute("SET hive.exec.dynamic.partition.mode = nonstrict")

# 3. Crear la tabla particionada como SEQUENCEFILE
create_partitioned_sql = """
CREATE TABLE IF NOT EXISTS sales_partitioned (
    id INT, 
    producto STRING, 
    categoria STRING, 
    monto DOUBLE
) 
PARTITIONED BY (pais STRING) 
STORED AS SEQUENCEFILE
"""
print("Creando tabla 'sales_partitioned'...")
cursor.execute(create_partitioned_sql)

# 4. Insertar los datos desde la tabla original
# Hive tomará la última columna del SELECT (pais) para crear las carpetas de partición
insert_sql = """
INSERT OVERWRITE TABLE sales_partitioned 
PARTITION(pais) 
SELECT id, producto, categoria, monto, pais 
FROM sales
"""
print("Insertando datos y creando particiones por país... (esto puede tardar un poco)")
cursor.execute(insert_sql)

print("¡Proceso completado con éxito!")

# 5. Cerrar conexión
conexion.close()

Creando tabla 'sales_partitioned'...
Insertando datos y creando particiones por país... (esto puede tardar un poco)
¡Proceso completado con éxito!


### Probando la estructura granular subyacente

Las particiones en Hive equivalen mágicamente a sub-directorios dentro de HDFS bajo la bandera `nombre_columna=valor`. ¡Verifiquémoslo nativamente desde el File System distribuido!


In [9]:
# Se guarda por defecto en el 'warehouse directory' configurado en {HIVE_DATADIR}
!docker exec namenode hdfs dfs -ls {HIVE_DATADIR}/external/lab_db/sales
!docker exec namenode hdfs dfs -ls {HIVE_DATADIR}/internal/lab_db.db/sales_partitioned/

Found 1 items
-rw-r--r--   3 hadoop hive    1853396 2026-04-06 06:52 /user/hive/warehouse/external/lab_db/sales/sales.csv
Found 4 items
drwxr-xr-x   - hive supergroup          0 2026-04-06 06:53 /user/hive/warehouse/internal/lab_db.db/sales_partitioned/pais=CO
drwxr-xr-x   - hive supergroup          0 2026-04-06 06:53 /user/hive/warehouse/internal/lab_db.db/sales_partitioned/pais=ES
drwxr-xr-x   - hive supergroup          0 2026-04-06 06:53 /user/hive/warehouse/internal/lab_db.db/sales_partitioned/pais=MX
drwxr-xr-x   - hive supergroup          0 2026-04-06 06:53 /user/hive/warehouse/internal/lab_db.db/sales_partitioned/pais=US
